### Note: The following notebook has been used to generate the plots for the website on RQ1 part. Since it had a lot of trial and error, it has a file on his own. The code is identical to that of the `results.ipyb` untill the genereated plot.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)


# Imports
from src.models.log_log_efficiency_model import linear_log_log_model
from config import (
    DATASET_CHANNELS,
    DATASET_TIMESERIES
)
from src.utils.plots import *

# Style
sns.set_style("whitegrid")
sns.set_palette("husl")
seed = 6 # for reproducing the results

%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [3]:
channel_df = pd.read_parquet(DATASET_CHANNELS)
channel_df.columns

Index(['category_cc', 'join_date', 'channel', 'name_cc', 'subscribers_cc',
       'videos_cc', 'subscriber_rank_sb', 'weights', 'channel_age_days',
       'videos_per_day', 'avg_engagement_efficiency', 'total_views',
       'total_duration', 'total_like_count', 'total_dislike_count',
       'total_num_comms', 'avg_video_like_ratio', 'video_count_processed'],
      dtype='object')

In [ ]:
# Saved previously - its the same df as the one achieved in results.ipynb point 4.
outperformers = pd.read_parquet(
    r"C:\Users\Jon\Desktop\ADA Project2\ada-2025-project-in5ight\data\cached_results\channels_outperformers.parquet"
)

In [5]:
timeseries_df = pd.read_parquet(DATASET_TIMESERIES)

In [6]:
channel_time_df = outperformers.merge(timeseries_df, how="inner", on="channel")
channel_time_df.head()

,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights,channel_age_days,videos_per_day,...,activity,week_start,efficiency_4week,efficiency_4week_filled,posting_pace_4w,posting_consistency_4w,posting_pace_8w,posting_consistency_8w,posting_pace_12w,posting_consistency_12w
0,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,...,14,2016-10-11,0.046551,0.046551,0.0,0.0000,0.0,0.000000,0.0,0.000000
1,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,...,14,2016-10-18,0.044049,0.044049,0.0,1.0000,0.0,1.000000,0.0,1.000000
2,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,...,15,2016-10-25,0.045417,0.045417,0.0,0.0000,0.0,0.000000,0.0,0.000000
3,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,...,15,2016-11-01,0.047641,0.047641,6.0,0.0000,6.0,0.000000,6.0,0.000000
4,Gaming,2010-04-29,UC-lHJZR3Gqxm24_Vd_AJ5Yw,PewDiePie,101000000,3956,3.0,2.087,3468.0,1.140715,...,14,2016-11-08,0.052416,0.052416,7.0,0.3267,6.0,0.081441,6.0,0.081441


In [7]:
channel_time_df.columns

Index(['category_cc', 'join_date', 'channel', 'name_cc', 'subscribers_cc',
       'videos_cc', 'subscriber_rank_sb', 'weights', 'channel_age_days',
       'videos_per_day', 'avg_engagement_efficiency', 'total_views',
       'total_duration', 'total_like_count', 'total_dislike_count',
       'total_num_comms', 'avg_video_like_ratio', 'video_count_processed',
       'is_outperformer', 'category', 'datetime', 'views', 'delta_views',
       'subs', 'delta_subs', 'videos', 'delta_videos', 'activity',
       'week_start', 'efficiency_4week', 'efficiency_4week_filled',
       'posting_pace_4w', 'posting_consistency_4w', 'posting_pace_8w',
       'posting_consistency_8w', 'posting_pace_12w',
       'posting_consistency_12w'],
      dtype='object')

In [8]:
small_mid_sized_channels_df = channel_df[channel_df["subscribers_cc"] < 100_000].copy()

In [9]:
# 1. Run the model on the static channel data 
# This identifies the global relationship: log(Efficiency) = a + b * log(Subscribers)
channels_model_df = linear_log_log_model(
    df=small_mid_sized_channels_df, # Your static dataset
    x_col="subscribers_cc",
    y_col="avg_engagement_efficiency",
    weight_col="weights",
    log_base=10,
    verbose=True,
)

# 2. Extract the slope (b) and intercept (a) from the custom attributes
model_params = channels_model_df.attrs["model"]
intercept_a = model_params["a"]
slope_b = model_params["b"]

print(f"Global Baseline Established: Intercept (a) = {intercept_a:.4f}, Slope (b) = {slope_b:.4f}")
print("Concept: As channel size grows, expected efficiency scales by a factor of", round(slope_b, 4))

log–log model fitted: log(y) = -0.9058 + -0.1419 * log(x)
Global Baseline Established: Intercept (a) = -0.9058, Slope (b) = -0.1419
Concept: As channel size grows, expected efficiency scales by a factor of -0.1419


In [10]:
# Data Prep (Creating the "Bridge" Data)

# 1. Select the specific outperformers
top_channel_ids = (
    outperformers
    .sort_values("avg_engagement_efficiency", ascending=False)
    .head(5)["channel"]
    .unique()
)

# 2. Filter the time-series dataframe
df_viz = channel_time_df[channel_time_df['channel'].isin(top_channel_ids)].copy()
df_viz['week_start'] = pd.to_datetime(df_viz['week_start'])
df_viz = df_viz.sort_values(['channel', 'week_start'])

# 3. Calculate Baseline
df_viz = df_viz.dropna(subset=['subs']) 
df_viz['log_subs_dynamic'] = np.log10(df_viz['subs'])
df_viz['log_expected_efficiency'] = intercept_a + (slope_b * df_viz['log_subs_dynamic'])
df_viz['expected_baseline'] = 10 ** df_viz['log_expected_efficiency']

# 4. Prepare the "Gap" Logic
# Ensure 0s are NaNs so they count as gaps
df_viz.loc[df_viz['efficiency_4week'] == 0, 'efficiency_4week'] = np.nan

# Create a NEW column that fills the gaps (for the dashed line)
# We use interpolate to draw a straight line between the existing points
df_viz['efficiency_bridged'] = df_viz.groupby('channel')['efficiency_4week'].transform(
    lambda x: x.interpolate(method='linear', limit_direction='both')
)

print("Data prepared: 'efficiency_4week' has gaps, 'efficiency_bridged' is full.")

Data prepared: 'efficiency_4week' has gaps, 'efficiency_bridged' is full.


In [11]:
outperformers = outperformers[outperformers["subscribers_cc"] < 100_000]

In [12]:
outperformers.sort_values("avg_engagement_efficiency", ascending=False).head(30)

,category_cc,join_date,channel,name_cc,subscribers_cc,videos_cc,subscriber_rank_sb,weights,channel_age_days,videos_per_day,avg_engagement_efficiency,total_views,total_duration,total_like_count,total_dislike_count,total_num_comms,avg_video_like_ratio,video_count_processed,is_outperformer
107491,People & Blogs,2013-08-15,UCDNlTovDJmWXXuu0Vr3dSkA,thestarsiens,19300,72,615445.0,9.9470,2264.0,0.031802,6.464628,16240.0,29267.0,5525.0,191.0,554.0,0.942757,65,True
82854,Music,2012-10-15,UCfhrFEDU1Rc-VW2I2kBE6jw,azrielstar1,29969,531,404769.0,8.2070,2568.0,0.206776,5.258212,131843.0,137013.0,33516.0,1214.0,3414.0,0.965670,529,True
103913,Gaming,2017-03-15,UCbM-liOrjvaPTg4f6mvOPjw,Yung Lil Dreamz,20900,79,582345.0,5.5840,956.0,0.082636,1.675526,7436.0,162775.0,10571.0,213.0,1265.0,0.869933,79,True
126440,Film and Animation,2016-06-15,UCsVymTrypChaL5jyvVeamsQ,Silomroad HollandAsi...,12900,89,812868.0,11.3355,1229.0,0.072417,1.033560,190371.0,12655.0,44980.0,199.0,140266.0,0.995239,90,True
105817,Gaming,2015-06-28,UCfs20O_RuLJXGGTxv5VIzdw,Gym Monk\Gaming,17100,962,601968.0,11.3885,1582.0,0.608091,0.756533,5066128.0,1596215.0,77333.0,3555.0,67211.0,0.992188,865,True
123179,Gaming,2017-08-18,UCt26eKdThkVbgRKO0C-EC-g,Variety TV,10500,16,780655.0,9.9735,800.0,0.020000,0.751197,14562.0,8484.0,12771.0,111.0,89.0,0.990563,16,True
133620,News & Politics,2017-07-30,UCr6fl90LzES3vTPVMG1Cu4A,The Marketplace,11200,78,911616.0,15.5355,819.0,0.095238,0.720728,444.0,66461.0,102.0,4.0,90.0,0.979853,92,True
77757,Gaming,2017-05-28,UCjUW4iEi9j-SPiCKXeZOeyA,NickNasty,21800,95,368977.0,8.7685,882.0,0.107710,0.658258,6208976.0,64999.0,70128.0,2301.0,2002.0,0.925841,92,True
46073,Entertainment,2017-04-13,UC-lexmz0-Rw_Xko2RfL45iw,Kids Love Playtime,95400,177,173196.0,4.3460,927.0,0.190939,0.647750,626350.0,123349.0,8955.0,1347.0,16591.0,0.926106,177,True
124605,Gaming,2015-12-22,UC9Q_Ej3wDtqOmR9X62vqSQQ,yogi gaming,10300,204,790550.0,10.5470,1405.0,0.145196,0.585438,71221.0,370262.0,12215.0,258.0,225.0,0.913245,203,True


In [13]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os

# 1. Setup
output_folder = "assets/channel_charts" 
os.makedirs(output_folder, exist_ok=True)

# 2. Define the Function
def create_fig(channel_id, raw_df, intercept, slope):
    # --- Data Prep ---
    clean_id = channel_id.strip()
    mask = raw_df['channel'].astype(str).str.strip() == clean_id
    df = raw_df[mask].copy()
    
    if df.empty: return None

    df['week_start'] = pd.to_datetime(df['week_start']).sort_values()
    df = df[df['subs'] > 0]
    df['expected_baseline'] = 10 ** (intercept + (slope * np.log10(df['subs'])))
    
    # Bridge Gaps
    if 'efficiency_4week' in df.columns:
        df.loc[df['efficiency_4week'] == 0, 'efficiency_4week'] = np.nan
        df['efficiency_bridged'] = df['efficiency_4week'].interpolate(method='linear', limit=2)
    else: return None

    name = df['name_cc'].iloc[0] if 'name_cc' in df.columns else clean_id

    # --- Plotting ---
    fig = go.Figure()
    
    # Trace 1: The Bridge -> Renamed to "Inactivity"
    fig.add_trace(go.Scatter(
        x=df['week_start'], 
        y=df['efficiency_bridged'], 
        mode='lines',
        name='Inactivity',
        line=dict(color='#44AA99', width=2, dash='dash'), 
        opacity=0.6,
        showlegend=True,   
        hoverinfo='skip'
    ))

    # Trace 2: Actual
    fig.add_trace(go.Scatter(
        x=df['week_start'], 
        y=df['efficiency_4week'], 
        mode='lines',
        line=dict(color='#44AA99', width=3), 
        name='Actual Efficiency'
    ))

    # Trace 3: Baseline
    fig.add_trace(go.Scatter(
        x=df['week_start'], 
        y=df['expected_baseline'], 
        mode='lines',
        line=dict(color='#882255', width=2, dash='dot'), 
        name='Expected Baseline'
    ))

    # --- Layout Fixes ---
    fig.update_layout(
        title=dict(text=f"{name}", x=0, xanchor='left'),
        template="plotly_white",
        hovermode="x unified",
        
        # FIX: Move Legend to Top (Horizontal) so it doesn't take side space
        legend=dict(
            orientation="h",   # Horizontal orientation
            yanchor="bottom", 
            y=1.02,            # Position slightly above the chart
            xanchor="left", 
            x=0                # Align left
        ),
        
        # FIX: Tight margins so the chart fills the iframe
        margin=dict(l=20, r=20, t=60, b=20) 
    )
    return fig

# 3. Batch Save
selected_ids = [
    "UCfs20O_RuLJXGGTxv5VIzdw", # Gym Monk
    "UC-lexmz0-Rw_Xko2RfL45iw", # Kids love playtime
    "UCVwhjRD688mQ4HAHakK5fYA", # Ellen
    "UCzDvtKtrxGU_V7kiyRnXqHQ", #ENdzow
    "UCOM38bKFwSWdkFMWHG5yiHw", #Super RC
    "UCfgrbR2muXpdANuDlv1Pyyg" , #Rainey TV
    "UCpZHrDn7nyGQoYl35rsr59Q", # THe craft corner
    "UCSK-WeqcoT8Ax3Q4hQUHXPg",# Tosh TIme,
    "UCyirgkrx41-iuDsgdDxIY3g" # TecnoFacil / STREAMIT Coin
]

print("Overwriting charts with new legend layout...")
for ch_id in selected_ids:
    fig = create_fig(ch_id, channel_time_df, intercept_a, slope_b)
    if fig:
        fig.show()
        filename = f"{output_folder}/{ch_id.strip()}.html"
        fig.write_html(filename, include_plotlyjs='cdn', full_html=False)
        print(f" Updated: {filename}")

Overwriting charts with new legend layout...


 Updated: assets/channel_charts/UCfs20O_RuLJXGGTxv5VIzdw.html


 Updated: assets/channel_charts/UC-lexmz0-Rw_Xko2RfL45iw.html


 Updated: assets/channel_charts/UCVwhjRD688mQ4HAHakK5fYA.html


 Updated: assets/channel_charts/UCzDvtKtrxGU_V7kiyRnXqHQ.html


 Updated: assets/channel_charts/UCOM38bKFwSWdkFMWHG5yiHw.html


 Updated: assets/channel_charts/UCfgrbR2muXpdANuDlv1Pyyg.html


 Updated: assets/channel_charts/UCpZHrDn7nyGQoYl35rsr59Q.html


 Updated: assets/channel_charts/UCSK-WeqcoT8Ax3Q4hQUHXPg.html


 Updated: assets/channel_charts/UCyirgkrx41-iuDsgdDxIY3g.html
